In [1]:
from util.coordinates import transform_rel_particle_coordinates_to_cartesian
import pickle
path = "downloaded_output/2025-08-14_22-51-39--519a204b-4be2-4481-8f67-9f1ebc94da01"
with open(f"{path}/data/x_test.pkl", "rb") as f:
    X_train = pickle.load(f)
X_train_particle_transformed = transform_rel_particle_coordinates_to_cartesian(X_train)
scale = open(f"{path}/train/scale.txt").read().strip()
scale = float(scale)
X_train_particle_transformed /= scale

/Users/arjunsharma/development/s25-fermilab-research/src/venv/lib/python3.10/site-packages/coffea/nanoevents/schemas/fcc.py:5: FutureWarning: In version 2025.1.0 (target date: 2024-12-31 11:59:59-06:00), this will be an error.
To raise these warnings as errors (and get stack traces to find out where they're called), run
    import warnings
    warnings.filterwarnings("error", module="coffea.*")
after the first `import coffea` or use `@pytest.mark.filterwarnings("error:::coffea.*")` in pytest.
Issue: coffea.nanoevents.methods.vector will be removed and replaced with scikit-hep vector. Nanoevents schemas internal to coffea will be migrated. Otherwise please consider using that package!.
  from coffea.nanoevents.methods import vector


FileNotFoundError: [Errno 2] No such file or directory: 'downloaded_output/2025-08-14_22-51-39--519a204b-4be2-4481-8f67-9f1ebc94da01/data/x_test.pkl'

In [ ]:
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment

def __optimal_permute(initial_x_0, target_x1):
    assert initial_x_0.shape == target_x1.shape
    cost = cdist(initial_x_0, target_x1, metric='euclidean')
    cost = cost * (10000000. / cost.max())
    row_ind, col_ind = linear_sum_assignment(cost)

In [ ]:
from util.distributions import gen_initial_distribution

x_1 = X_train_particle_transformed[:100]
x_1 = x_1[:, :, :4]
x_0 = gen_initial_distribution(x_1=x_1)

In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation as R

def align_point_clouds_till_converge(x_0_orig, x1, max_iter=2_500):
    x_0 = x_0_orig.clone()
    i = 0
    dist = np.linalg.norm(x_0 - x1, axis=1).sum()
    dist_delta = np.inf
    while i < max_iter and dist_delta > 1e-8:
        cost = cdist(x_0, x1, metric='euclidean')
        # Better for stability
        cost = cost * (1_000. / cost.max())
        _, col_ind = linear_sum_assignment(cost)

        x_0 = x_0[col_ind]
        # Align cartesian 3-momenta
        rot, _, _ = R.align_vectors(x1[:, 1:4], x_0[:, 1:4], return_sensitivity=True)
        x_0[:, 1:4] = x_0[:, 1:4] @ rot.as_matrix().T

        dist_new = np.linalg.norm(x_0 - x1, axis=1).sum()
        dist_delta = np.abs(dist_new - dist)
        dist = dist_new
        i += 1
    print(f"{i=} {dist=} {dist_delta=}")

In [ ]:
for j in range(48):
    %time align_point_clouds_till_converge(x_0[j], x_1[j])